# GP01C Qualification Retry

Recheck the development cases with outstanding infrastructure issues.

This source copy comes from **gp01c.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

# GP01C — Resolve the two remaining development qualification cases

**Clifford Imaguezegie · Gemma 4 paper study · 5 November 2026 · v1.0**

GP01B revision 1.2 executed 11 pytest runs with zero model calls. Four tasks qualified in fresh baseline/reference sandboxes: `httpx_3672` (24/24 reference tests passed), `rich_2943` (27/27), `fastapi_5624` (1/1), and `fastapi_14479` (4/4). Their actual module origins were inside the corresponding snapshots, and paired archive hashes/native Git identities matched. Those four are retained as completed development qualification evidence and are not rerun by this notebook. They are not coding-agent results.

| Remaining task | Observed issue | This follow-up |
| --- | --- | --- |
| requests_6757 | Baseline collection cannot import `is_urllib3_1` from snapshot `requests.compat`. | Run both phases for diagnosis and record whether the reference patch addresses the missing symbol. The strict qualification rule still rejects a baseline collection error. |
| rich_3130 | The reference fix resolves the new table/style tests, but two existing syntax-colour tests still fail under Pygments 2.21.0. | Test a single recorded compatibility hypothesis: Pygments 2.15.1 in both task phases. Preserve the official test file and expectations. |

The Rich errors show colour changes (`#f92672` versus `#ff4689`) and whitespace token colouring. The [Pygments changelog](https://pygments.org/docs/changelog/) documents style changes, supporting a dependency-compatibility hypothesis. Pygments **2.15.1 is a fixed development probe**, not a claim that it is the benchmark author's exact original environment. Its wheel and installed file hashes are retained. The older dependency is placed only in the Rich task sandbox through a separate overlay; the evaluator runtime and Requests case keep their existing packages. If it does not restore the reference pass, the failure remains in the results. All previous attempts remain valid development history; nothing is erased or relabelled as success.

The official verifier, protected-test rules, test targets, source code and supplied reference patches are unchanged. Archives remain pinned to the hashes returned by GP01B. Task-specific dependency profiles apply identically to baseline/reference now and must apply identically to future A/B/C arms. No model is loaded or called. No evaluation-candidate task is executed, and no final split is frozen. Detailed test evidence remains evaluator-only.

## Run instructions

1. Import this notebook and keep the **Developer Agent competition** and **metric wheelhouse** inputs available.
2. Select **CPU / None** and enable **Internet** for the pinned dependency download or any missing evaluator packages. No API key or GPU is needed.
3. Run **Cells 1–6**. Only `rich_3130` and `requests_6757` are executed. Both phases are deliberately run even if baseline collection fails; that is diagnostic, never automatic qualification.
4. Return **GP01C_results.zip**. Cell 6 attempts a browser download and provides a Download ZIP button/file link when notebook permissions allow. The Output panel remains the fallback.

We will use this result to close or document the remaining holds, then prepare the first autonomous repair smoke run using an already-qualified task. Package installation and these actual Kaggle cases have not been executed in the authoring environment.

**Local validation:** Seven checks passed, covering compilation, targeted selection, real Python dependency-overlay routing, hash-tamper rejection, unchanged collection-error qualification, diagnostic ZIP export and preserved official-verifier guards. The actual pinned Pygments package and Kaggle task outcomes still require this execution.


## Cell 1 — Configure the qualification run and diagnostic output


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import runpy
import shutil
import signal
import subprocess
import sys
import time
import uuid
import venv
import zipfile

COMPETITION_ROOT = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_ROOT = None  # Existing mount is detected; KaggleHub is the fallback.
TASK_IDS = ['rich_3130', 'requests_6757']
STOP_ON_INFRA_HOLD = False
ALLOW_PYPI_BOOTSTRAP = True
AUTO_DOWNLOAD_RESULTS = True
COMMAND_TIMEOUT_SECONDS = 300
VERIFICATION_TIMEOUT_SECONDS = 300
PHASE_WALL_TIMEOUT_SECONDS = 1800

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = Path('/kaggle/working/Gemma4_Paper') / ('GP01C_' + RUN_ID)
ARTIFACTS = RUN_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=False)
HOLDS, OBSERVATIONS, TASK_RESULTS = [], [], []
RUNTIME_READY = False
VENV_PYTHON = None

def file_sha(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

def write_json(name, value):
    (ARTIFACTS / name).write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')

def observe(cell, message, **details):
    entry = {'cell': cell, 'observation': message, **details}
    OBSERVATIONS.append(entry)
    print(json.dumps(entry, indent=2), flush=True)

def terminate_tree(process):
    # Limit cleanup to descendants of the worker created by this notebook.
    try:
        listing = subprocess.run(['ps', '-eo', 'pid=,ppid='], capture_output=True, text=True, timeout=10)
        pairs = [tuple(map(int, line.split())) for line in listing.stdout.splitlines() if len(line.split()) == 2]
        descendants, frontier = [], {process.pid}
        while frontier:
            children = {pid for pid, parent in pairs if parent in frontier} - set(descendants)
            descendants.extend(children)
            frontier = children
        for pid in reversed(descendants):
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
    finally:
        if process.poll() is None:
            process.kill()
        process.wait(timeout=15)

def run_logged(command, log_name, timeout_seconds):
    destination = ARTIFACTS / log_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    last_notice = started
    with destination.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                   text=True, start_new_session=True)
        while process.poll() is None:
            now = time.monotonic()
            if now - started > timeout_seconds:
                terminate_tree(process)
                log.write('\nGP01B: worker wall-time limit exceeded.\n')
                return {'returncode': -9, 'timed_out': True, 'duration_seconds': round(now-started, 2)}
            if now - last_notice >= 30:
                print(log_name + ': running (' + str(int(now-started)) + ' seconds)', flush=True)
                last_notice = now
            time.sleep(1)
        return {'returncode': process.returncode, 'timed_out': False,
                'duration_seconds': round(time.monotonic()-started, 2)}

observe(1, 'Created a separate CPU qualification run; no model will be invoked.', output=str(RUN_DIR))


**Observation:** A new run directory is printed. Keep the two task IDs unchanged for this first execution.


## Cell 2 — Prepare a compatible evaluator runtime from the mounted wheels


In [ ]:
CORE_PINS = [{'distribution': 'adk-eval-core', 'version': '0.1.0', 'sha256': 'e614e1b2fe8d886a634871e4c57f9887e1cc66e8c27300f8503a43c6e3ca82f3'}, {'distribution': 'adk-submission', 'version': '0.2.13', 'sha256': 'f156f0bd84580ccb9334726d114d7a5481f16d3a6e2887cc4d276ced0ef8d99d'}, {'distribution': 'swegemma', 'version': '0.2.11', 'sha256': '9ad876581f3acde8ec3c369858f0834e2b9f41a3cffce3ca42f09cb090aea9d3'}]
DEVELOPMENT = [{'instance_id': 'fastapi_14479', 'repo': 'fastapi/fastapi', 'base_commit': '2b212ddd7604891d42e2ffbafe764c334230bdb7', 'split_group': 'fastapi/fastapi@2b212ddd7604891d42e2ffbafe764c334230bdb7', 'group_role': 'development_candidate'}, {'instance_id': 'fastapi_5624', 'repo': 'fastapi/fastapi', 'base_commit': 'c6487ed632056e450d844846a1b63be551a3cbc6', 'split_group': 'fastapi/fastapi@c6487ed632056e450d844846a1b63be551a3cbc6', 'group_role': 'development_candidate'}, {'instance_id': 'httpx_3672', 'repo': 'encode/httpx', 'base_commit': '4acf5c2c37714cc63b5cf71b3e284fca83c90311', 'split_group': 'encode/httpx@4acf5c2c37714cc63b5cf71b3e284fca83c90311', 'group_role': 'development_candidate'}, {'instance_id': 'requests_6757', 'repo': 'psf/requests', 'base_commit': '0e322af87745eff34caffe4df68456ebc20d9068', 'split_group': 'psf/requests@0e322af87745eff34caffe4df68456ebc20d9068', 'group_role': 'development_candidate'}, {'instance_id': 'rich_2943', 'repo': 'Textualize/rich', 'base_commit': '610fd75c433c3dfcd9c296e8b91363979127b9fa', 'split_group': 'Textualize/rich@610fd75c433c3dfcd9c296e8b91363979127b9fa', 'group_role': 'development_candidate'}, {'instance_id': 'rich_3130', 'repo': 'Textualize/rich', 'base_commit': '720800e6930d85ad027b1e9bd0cbb96b5e994ce3', 'split_group': 'Textualize/rich@720800e6930d85ad027b1e9bd0cbb96b5e994ce3', 'group_role': 'development_candidate'}]
try:
    from packaging.version import Version
    if not (COMPETITION_ROOT / 'tasks.jsonl').is_file():
        raise FileNotFoundError('Attach the Developer Agent competition input.')
    if file_sha(COMPETITION_ROOT / 'tasks.jsonl') != 'e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6':
        raise RuntimeError('The task release differs from GP00/GP01A; review the change.')
    known_mounts = [Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse'),
                    Path('/kaggle/input/gemma-4-developer-agent-wheelhouse')]
    if WHEELHOUSE_ROOT is None:
        WHEELHOUSE_ROOT = next((p for p in known_mounts if p.is_dir()), None)
    if WHEELHOUSE_ROOT is None:
        import kagglehub
        WHEELHOUSE_ROOT = Path(kagglehub.dataset_download('metric/gemma-4-developer-agent-wheelhouse'))
    WHEELHOUSE_ROOT = Path(WHEELHOUSE_ROOT)
    core_files, builds = [], []
    for pin in CORE_PINS:
        prefix = pin['distribution'].replace('-', '_') + '-'
        options = [p for p in WHEELHOUSE_ROOT.rglob('*.whl') if p.name.startswith(prefix)]
        exact = [p for p in options if file_sha(p) == pin['sha256']]
        if len(exact) != 1:
            raise RuntimeError('Expected exactly one GP01A-matching wheel for ' + pin['distribution'] + '; inspect the wheelhouse release.')
        core_files.append(exact[0])
        builds.append({**pin, 'path': str(exact[0])})
    write_json('01_core_builds.json', builds)
    adk_options = [(Version(p.name.split('-')[1]), p) for p in WHEELHOUSE_ROOT.rglob('google_adk-*.whl')]
    adk_options = [(version, path) for version, path in adk_options if Version('1.34.0') <= version < Version('2')]
    if not adk_options:
        raise RuntimeError('No compatible Google ADK 1.x wheel was found in the wheelhouse.')
    adk_version, adk_wheel = max(adk_options, key=lambda item: item[0])
    environment = RUN_DIR / 'runtime_venv'
    try:
        venv.create(environment, system_site_packages=True, with_pip=True)
    except Exception:
        venv.create(environment, system_site_packages=True, with_pip=False, clear=True)
    VENV_PYTHON = environment / 'bin/python'
    available = [p for p in (COMPETITION_ROOT / 'wheels').glob('*.whl')]
    optional_test_packages = ['pytest', 'pytest-cov', 'pytest-mock', 'pytest-timeout',
                              'pytest-asyncio', 'pytest-trio', 'hypothesis', 'trio']
    test_packages = [name for name in optional_test_packages
                     if any(p.name.lower().startswith(name.replace('-', '_') + '-') for p in available)]
    core_install = [str(VENV_PYTHON), '-m', 'pip', '--disable-pip-version-check', 'install',
               '--require-virtualenv', '--no-index', '--only-binary=:all:',
               '--no-deps', *map(str, core_files)]
    print('Preparing the isolated evaluator runtime from local wheels...', flush=True)
    core_installation = run_logged(core_install, 'core_install.log', 300)
    if core_installation['returncode'] != 0:
        raise RuntimeError('Core-wheel installation failed. See core_install.log.')
    runtime_requirements = [str(adk_wheel), 'cachetools>=5.0', 'docker>=7.0', 'google-genai>=1.0',
                            'networkx>=3.0', 'numpy>=2.0', 'pydantic>=2.0', 'python-dotenv>=1.0',
                            'pyyaml>=6.0', 'rich>=13.0', 'litellm>=1.0', 'pandas>=2.0', *test_packages]
    install = [str(VENV_PYTHON), '-m', 'pip', '--disable-pip-version-check', 'install',
               '--require-virtualenv', '--no-index', '--only-binary=:all:',
               '--find-links', str(WHEELHOUSE_ROOT), '--find-links', str(COMPETITION_ROOT / 'wheels'),
               *runtime_requirements]
    installation = run_logged(install, 'runtime_install_offline.log', 1200)
    network_fallback_used = installation['returncode'] != 0 and ALLOW_PYPI_BOOTSTRAP
    if network_fallback_used:
        print('Some CPU runtime dependencies are unavailable offline; resolving them from PyPI in the separate environment.', flush=True)
        install = [arg for arg in install if arg != '--no-index']
        installation = run_logged(install, 'runtime_install_pypi.log', 1200)
    write_json('02_runtime_install.json', {'result': installation, 'command': install,
               'core_installation': core_installation, 'core_install_command': core_install,
               'network_fallback_used': network_fallback_used,
               'google_adk_wheel': {'path': str(adk_wheel), 'version': str(adk_version), 'sha256': file_sha(adk_wheel)},
               'inherits_existing_host_packages': True, 'test_packages_requested': test_packages,
               'wheelhouse_root': str(WHEELHOUSE_ROOT)})
    if installation['returncode'] != 0:
        raise RuntimeError('Runtime dependency installation failed. See the runtime_install logs in the output ZIP.')
    observe(2, 'Prepared a separate runtime using the three fingerprinted core builds and google-adk<2.',
            wheelhouse=str(WHEELHOUSE_ROOT), installation=installation)
except Exception as exc:
    HOLDS.append(type(exc).__name__ + ': ' + str(exc))
    observe(2, 'Runtime preparation is on hold. Continue to Cells 3–6 to export diagnostics.', holds=HOLDS)


**Observation:** Installation occurs inside the new virtual environment. If it is on hold, continue through Cell 6; do not install arbitrary replacement versions.


## Cell 3 — Write the worker and verify its imports and exact source contract


In [ ]:
WORKER_SOURCE = '"""GP01B evaluator-only worker; never invokes a coding agent or model."""\nfrom __future__ import annotations\n\nimport argparse\nimport asyncio\nimport dataclasses\nimport hashlib\nimport importlib.metadata\nimport inspect\nimport json\nimport os\nimport re\nimport shlex\nimport sys\nimport time\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + \'\\n\')\n    temporary.replace(path)\n\n\ndef prepare_snapshot_workspace(original_exec, manager, sandbox_id, task, artifact_dir, compatibility=None):\n    """Use the supplied archive unchanged and put its source ahead of installed packages."""\n    paths = manager.sandboxes[sandbox_id]\n    workspace = Path(paths[\'workspace\']).resolve()\n    package = task.repo.rsplit(\'/\', 1)[-1].replace(\'-\', \'_\')\n    diagnostic = {\'declared_base_commit\': task.base_commit, \'workspace\': str(workspace),\n                  \'package\': package, \'status\': \'PREPARING\'}\n    diagnostic_path = Path(artifact_dir) / \'workspace_setup.json\'\n\n    def execute(command):\n        return original_exec(sandbox_id, command, timeout=60)\n\n    def fail(message):\n        diagnostic[\'status\'] = \'WORKSPACE_HOLD\'\n        diagnostic[\'error\'] = message\n        write_json(diagnostic_path, diagnostic)\n        raise RuntimeError(message)\n\n    before = execute(\'cd /workspace && git rev-parse HEAD\')\n    diagnostic[\'snapshot_head_before\'] = (before.stdout or \'\').strip()\n    diagnostic[\'root_entries_before\'] = sorted(p.name for p in workspace.iterdir())[:100]\n    if before.exit_code != 0 or not diagnostic[\'snapshot_head_before\']:\n        fail(\'The published snapshot has no readable Git HEAD; inspect its setup diagnostics.\')\n    commit = shlex.quote(task.base_commit + \'^{commit}\')\n    exists = execute(\'cd /workspace && git cat-file -e \' + commit)\n    diagnostic[\'declared_commit_exists\'] = exists.exit_code == 0\n    diagnostic[\'declared_commit_id_matches_head\'] = diagnostic[\'snapshot_head_before\'] == task.base_commit\n    tree = execute(\'cd /workspace && git rev-parse HEAD^{tree}\')\n    diagnostic[\'snapshot_git_tree\'] = (tree.stdout or \'\').strip()\n    if tree.exit_code != 0 or not diagnostic[\'snapshot_git_tree\']:\n        fail(\'The published snapshot has no readable Git tree.\')\n    diagnostic[\'identity_policy\'] = \'task_named_published_archive_sha256_and_recorded_native_git_identity\'\n    diagnostic[\'original_commit_id_required\'] = False\n    # The official verifier consumes the packaged snapshot directly. Its rebuilt\n    # Git history need not retain the original repository\'s commit identifiers.\n    # Do not fetch another repository or rewrite the supplied working tree.\n\n    candidates = []\n    for root in [workspace, workspace / \'src\']:\n        if (root / package / \'__init__.py\').is_file() or (root / (package + \'.py\')).is_file():\n            candidates.append(root)\n    diagnostic[\'source_roots_found\'] = [str(p) for p in candidates]\n    if len(candidates) != 1:\n        fail(\'Expected one explicit root/src source location for the repository package; inspect workspace_setup.json.\')\n    source_root = candidates[0]\n    environment = Path(paths[\'venv\']).resolve()\n    site_dirs = {p.resolve() for p in environment.glob(\'lib/python*/site-packages\') if p.is_dir()}\n    if len(site_dirs) != 1:\n        fail(\'Could not identify the sandbox virtual environment site-packages directory.\')\n    site_dir = next(iter(site_dirs))\n    if not site_dir.is_relative_to(environment):\n        fail(\'Sandbox source-path setup would escape its own virtual environment.\')\n    overlay = site_dir / \'zz_gp01b_snapshot_source.pth\'\n    source_paths = [str(source_root)]\n    if compatibility:\n        dependency_dir = Path(compatibility[\'path\']).resolve()\n        required_files = compatibility[\'file_hashes\']\n        for relative, expected in required_files.items():\n            selected = dependency_dir / relative\n            if not selected.is_file() or digest(selected) != expected:\n                fail(\'The pinned dependency overlay changed or is missing: \' + relative)\n        source_paths.append(str(dependency_dir))\n        diagnostic[\'compatibility_profile\'] = compatibility\n    overlay.write_text(\'import sys; sys.path[:0] = \' + repr(source_paths) + \'\\n\', encoding=\'utf-8\')\n    diagnostic[\'source_root\'] = str(source_root)\n    diagnostic[\'source_path_file\'] = str(overlay)\n    diagnostic[\'status\'] = \'WORKSPACE_READY\'\n    write_json(diagnostic_path, diagnostic)\n    return diagnostic\n\n\ndef verify_snapshot_archive(path, expected_sha256):\n    actual = digest(path)\n    if not expected_sha256 or actual != expected_sha256:\n        raise RuntimeError(\'Published snapshot bytes differ from the pinned task archive; review the dataset release.\')\n    return actual\n\n\ndef check_import_origin(original_exec, sandbox_id, package, workspace):\n    script = \'\'\'import importlib, importlib.util, json, sys\nfrom pathlib import Path\nname = sys.argv[1]\nspec = importlib.util.find_spec(name)\nout = {"package": name, "spec_origin": str(spec.origin) if spec else None}\ntry:\n    module = importlib.import_module(name)\n    out["imported_file"] = str(getattr(module, "__file__", None))\n    out["version"] = str(getattr(module, "__version__", "unknown"))\n    if name == "rich":\n        import pygments\n        out["dependency_probe"] = {"Pygments": {"version": pygments.__version__, "file": pygments.__file__}}\nexcept Exception as exc:\n    out["import_error"] = type(exc).__name__ + ": " + str(exc)\nprint(json.dumps(out))\n\'\'\'\n    command = \'python3 -s -c \' + shlex.quote(script) + \' \' + shlex.quote(package)\n    result = original_exec(sandbox_id, command, timeout=60)\n    try:\n        data = json.loads((result.stdout or \'\').splitlines()[-1])\n    except (ValueError, IndexError):\n        data = {\'import_error\': \'Origin probe did not return JSON.\', \'stderr\': result.stderr or \'\'}\n    actual = data.get(\'imported_file\')\n    data[\'inside_snapshot\'] = bool(actual and Path(actual).resolve().is_relative_to(Path(workspace).resolve()))\n    data[\'probe_exit_code\'] = result.exit_code\n    return data\n\n\ndef junit_summary(text):\n    """Require actual testcase records, not only a successful process exit."""\n    out = {\'valid_xml\': False, \'tests\': 0, \'passed\': 0, \'failures\': 0,\n           \'errors\': 0, \'skipped\': 0, \'failed_nodes\': []}\n    try:\n        root = ET.fromstring(text)\n    except (ET.ParseError, TypeError):\n        return out\n    suites = [root] if root.tag == \'testsuite\' else list(root.findall(\'.//testsuite\'))\n    if not suites:\n        return out\n    cases = list(root.findall(\'.//testcase\'))\n    out[\'valid_xml\'] = True\n    out[\'tests\'] = len(cases)\n    for case in cases:\n        name = case.attrib.get(\'name\', \'\')\n        classname = case.attrib.get(\'classname\', \'\')\n        node = classname + \'::\' + name if classname else name\n        if case.find(\'error\') is not None:\n            out[\'errors\'] += 1\n        elif case.find(\'failure\') is not None:\n            out[\'failures\'] += 1\n            out[\'failed_nodes\'].append(node)\n        elif case.find(\'skipped\') is not None:\n            out[\'skipped\'] += 1\n        else:\n            out[\'passed\'] += 1\n    # Collection errors can also appear only in suite counters.\n    try:\n        out[\'errors\'] = max(out[\'errors\'], sum(int(s.attrib.get(\'errors\', 0)) for s in suites))\n    except ValueError:\n        out[\'valid_xml\'] = False\n    return out\n\n\ndef classify_pair(baseline, reference):\n    if not baseline or not reference:\n        return {\'qualified\': False, \'status\': \'PENDING\', \'reason\': \'Both phases are required.\'}\n    for phase in (baseline, reference):\n        if phase.get(\'worker_error\') or phase.get(\'pytest_calls\') != 1:\n            return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'A phase did not reach exactly one pytest execution.\'}\n        if not phase.get(\'junit\', {}).get(\'valid_xml\') or phase[\'junit\'].get(\'errors\', 0) > 0:\n            return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'Missing/malformed JUnit or collection/setup errors.\'}\n        if phase.get(\'baseline_commit_exit_code\') != 0 or not phase.get(\'snapshot_identity_verified\'):\n            return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'Published archive identity or baseline commit was not verified.\'}\n        if not all(phase.get(\'workspace_setup\', {}).get(k) for k in (\'snapshot_head_before\', \'snapshot_git_tree\')):\n            return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'Native Git snapshot identity was not recorded.\'}\n        if not phase.get(\'import_origin\', {}).get(\'inside_snapshot\'):\n            return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'The repository package was not imported from the active snapshot.\'}\n    if baseline.get(\'test_exit_code\') == 0:\n        return {\'qualified\': False, \'status\': \'BASELINE_ALREADY_PASSES\', \'reason\': \'The unmodified snapshot did not reproduce a failing test.\'}\n    if baseline.get(\'test_exit_code\') != 1 or baseline[\'junit\'].get(\'failures\', 0) < 1:\n        return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'The baseline did not produce ordinary test failures with pytest exit 1.\'}\n    if not baseline.get(\'required_failure_matched\'):\n        return {\'qualified\': False, \'status\': \'REPRODUCTION_REVIEW\', \'reason\': \'No baseline failure was matched to a required verification test.\'}\n    rj = reference[\'junit\']\n    if (not reference.get(\'resolved\') or reference.get(\'test_exit_code\') != 0\n            or rj.get(\'passed\', 0) < 1 or rj.get(\'failures\', 0) or rj.get(\'errors\', 0)\n            or not reference.get(\'official_junit_ok\')):\n        return {\'qualified\': False, \'status\': \'REFERENCE_NOT_VERIFIED\', \'reason\': \'The reference fix did not satisfy both the official verifier and independent JUnit checks.\'}\n    if baseline.get(\'snapshot_sha256\') != reference.get(\'snapshot_sha256\'):\n        return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'The phases used different snapshot bytes.\'}\n    for key in (\'snapshot_head_before\', \'snapshot_git_tree\'):\n        if baseline.get(\'workspace_setup\', {}).get(key) != reference.get(\'workspace_setup\', {}).get(key):\n            return {\'qualified\': False, \'status\': \'INFRA_HOLD\', \'reason\': \'The phases used different native Git snapshot identities.\'}\n    return {\'qualified\': True, \'status\': \'QUALIFIED\', \'reason\': \'Baseline required-test failure and reference-fix pass were observed in separate fresh sandboxes.\'}\n\n\ndef load_official(config):\n    # These switches prevent optional model metadata downloads during imports.\n    os.environ.setdefault(\'LITELLM_LOCAL_MODEL_COST_MAP\', \'True\')\n    os.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\')\n    os.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\n    from adk_submission import ModelRegistry\n    from swegemma.config import EvalConfig\n    from swegemma.models import load_tasks\n    from swegemma.sandbox import SubprocessManager\n    from swegemma.harness import verification\n    source_path = Path(inspect.getsourcefile(verification.verify_task))\n    actual_hash = digest(source_path)\n    if actual_hash != config[\'verification_source_sha256\']:\n        raise RuntimeError(\'Installed verification.py differs from the GP01A-inspected source.\')\n    parameters = list(inspect.signature(verification.verify_task).parameters)\n    if parameters != config[\'verify_task_parameters\']:\n        raise RuntimeError(\'Verifier signature differs from the inspected contract.\')\n    from packaging.version import Version\n    adk_version = importlib.metadata.version(\'google-adk\')\n    if not Version(\'1.34.0\') <= Version(adk_version) < Version(\'2\'):\n        raise RuntimeError(\'The isolated runtime must use google-adk>=1.34,<2.\')\n    return ModelRegistry, EvalConfig, load_tasks, SubprocessManager, verification\n\n\ndef preflight(config):\n    api = load_official(config)\n    names = {d.metadata[\'Name\'] for d in importlib.metadata.distributions() if d.metadata.get(\'Name\')}\n    package_versions = {name: importlib.metadata.version(name) for name in names}\n    return {\'status\': \'RUNTIME_READY\', \'python\': sys.version.split()[0],\n            \'verification_source_sha256\': config[\'verification_source_sha256\'],\n            \'verify_task_signature\': str(inspect.signature(api[-1].verify_task)),\n            \'packages\': dict(sorted(package_versions.items())), \'model_calls\': 0}\n\n\nasync def run_phase(config, task_id, phase):\n    ModelRegistry, EvalConfig, load_tasks, SubprocessManager, verification = load_official(config)\n    data_root = Path(config[\'competition_root\'])\n    if digest(data_root / \'tasks.jsonl\') != config[\'tasks_sha256\']:\n        raise RuntimeError(\'Task release changed since GP00/GP01A.\')\n    tasks = {t.instance_id: t for t in load_tasks(data_root / \'tasks.jsonl\')}\n    if task_id not in config[\'development_task_ids\']:\n        raise RuntimeError(\'GP01B is restricted to the six predeclared development candidates.\')\n    raw_task = tasks[task_id]\n    expected = config[\'development_metadata\'][task_id]\n    if (raw_task.repo, raw_task.base_commit) != (expected[\'repo\'], expected[\'base_commit\']):\n        raise RuntimeError(\'Task identity differs from the GP00 candidate manifest.\')\n    reference_patch = raw_task.patch\n    task = dataclasses.replace(raw_task, patch=\'\', hints_text=\'\')\n    del raw_task, tasks\n    snapshot = data_root / \'snapshots\' / (task_id + \'.tgz\')\n    snapshot_hash = verify_snapshot_archive(snapshot, config[\'snapshot_pins\'].get(task_id))\n    artifact_dir = Path(config[\'artifact_dir\']) / \'evaluator_only\' / task_id / phase\n    artifact_dir.mkdir(parents=True, exist_ok=True)\n    manager = SubprocessManager(timeout_seconds=config[\'command_timeout_seconds\'],\n                                base_dir=Path(config[\'sandbox_dir\']), system_site_packages=True)\n    manager.base_dir.mkdir(parents=True, exist_ok=True)\n    original_exec = manager.exec\n    test_events = []\n    git_setup = {}\n    workspace_setup = None\n    import_origin = {}\n\n    def recorded_exec(sandbox_id, command, *, timeout=None):\n        nonlocal workspace_setup, import_origin\n        # This is the first official setup command after snapshot extraction,\n        # before any test configuration or reference patch has been applied.\n        if workspace_setup is None and \'.git/info/exclude\' in command:\n            workspace_setup = prepare_snapshot_workspace(original_exec, manager, sandbox_id, task, artifact_dir, compatibility=config.get(\'dependency_overlays\', {}).get(task_id))\n        if \'git commit -m "eval_baseline"\' in command:\n            # The sanitized subprocess HOME has no global Git identity. Set only\n            # sandbox-local author metadata, identically in both phases.\n            identity = original_exec(sandbox_id,\n                \'cd /workspace && git config user.name GP01B && git config user.email gp01b@example.invalid && git rev-parse HEAD\', timeout=30)\n            git_setup[\'snapshot_head_matches\'] = identity.exit_code == 0 and (identity.stdout or \'\').strip() == task.base_commit\n            git_setup[\'native_snapshot_head_unchanged\'] = bool(workspace_setup and identity.exit_code == 0\n                and (identity.stdout or \'\').strip() == workspace_setup[\'snapshot_head_before\'])\n        is_pytest = bool(re.search(r\'python3\\s+-s\\s+-m\\s+pytest\\b\', command) and \'--junitxml=\' in command)\n        if is_pytest:\n            if workspace_setup is None:\n                raise RuntimeError(\'Snapshot setup did not run before pytest; inspect the harness setup sequence.\')\n            import_origin = check_import_origin(original_exec, sandbox_id, task.repo_name.replace(\'-\', \'_\'), manager.sandboxes[sandbox_id][\'workspace\'])\n            write_json(artifact_dir / \'import_origin.json\', import_origin)\n            compatibility = config.get(\'dependency_overlays\', {}).get(task_id)\n            if compatibility:\n                dependency = import_origin.get(\'dependency_probe\', {}).get(\'Pygments\', {})\n                if dependency.get(\'version\') != compatibility[\'version\'] or not dependency.get(\'file\') or not Path(dependency[\'file\']).resolve().is_relative_to(Path(compatibility[\'path\']).resolve()):\n                    raise RuntimeError(\'The Pygments compatibility profile was not loaded in the task sandbox.\')\n            if import_origin.get(\'import_error\') or not import_origin.get(\'inside_snapshot\'):\n                raise RuntimeError(\'Repository import did not resolve inside the checked-out snapshot. See import_origin.json.\')\n        result = original_exec(sandbox_id, command, timeout=timeout)\n        if \'git commit -m "eval_baseline"\' in command:\n            git_setup[\'baseline_commit_exit_code\'] = result.exit_code\n        if is_pytest:\n            xml_match = re.search(r\'--junitxml=([^\\s]+)\', command)\n            xml_text = \'\'\n            if xml_match:\n                xml_path = xml_match.group(1)\n                xml_result = original_exec(sandbox_id, \'cat \' + shlex.quote(xml_path) + \' 2>/dev/null || true\', timeout=30)\n                xml_text = xml_result.stdout or \'\'\n            serial = len(test_events) + 1\n            (artifact_dir / f\'pytest_{serial}.xml\').write_text(xml_text, encoding=\'utf-8\')\n            (artifact_dir / f\'pytest_{serial}.log\').write_text(\n                \'COMMAND: \' + command + \'\\nSTDOUT:\\n\' + (result.stdout or \'\') + \'\\nSTDERR:\\n\' + (result.stderr or \'\'), encoding=\'utf-8\')\n            test_events.append({\'exit_code\': result.exit_code, \'xml\': xml_text})\n        return result\n\n    # Instance instrumentation preserves the official SubprocessManager class identity\n    # and passes each command and return value through unchanged.\n    manager.exec = recorded_exec\n    eval_config = EvalConfig(tasks_path=data_root / \'tasks.jsonl\', snapshots_dir=data_root / \'snapshots\',\n                             results_dir=artifact_dir, submission_dir=artifact_dir / \'unused_submission\',\n                             models=ModelRegistry(), sandbox=\'subprocess\', wheels_dir=data_root / \'wheels\',\n                             timeout_seconds=config[\'command_timeout_seconds\'],\n                             verification_timeout_seconds=config[\'verification_timeout_seconds\'], verbose=False)\n    started = time.perf_counter()\n    try:\n        result = await verification.verify_task(manager, eval_config, task, snapshot,\n                                               agent_patch=reference_patch if phase == \'reference\' else \'\',\n                                               start_time=started)\n        xml_text = test_events[-1][\'xml\'] if test_events else \'\'\n        parsed = junit_summary(xml_text)\n        official_ok, official_reason = verification._validate_junit_xml(\n            xml_text, fail_to_pass=task.FAIL_TO_PASS, pass_to_pass=task.PASS_TO_PASS, test_patch=task.test_patch)\n        required = list(task.FAIL_TO_PASS) or verification._extract_test_functions_from_patch(task.test_patch)\n        matched = any(verification._node_matches_requirement(node, req, req.split(\'::\')[-1], req.split(\'::\')[-1].split(\'[\', 1)[0])\n                      for node in parsed[\'failed_nodes\'] for req in required)\n        payload = {\'instance_id\': task_id, \'repo\': task.repo, \'phase\': phase,\n                   \'snapshot_sha256\': snapshot_hash, \'reference_patch_sha256\': hashlib.sha256(reference_patch.encode()).hexdigest(),\n                   \'resolved\': bool(result.resolved), \'test_exit_code\': result.test_exit_code,\n                   \'error\': getattr(result, \'error_message\', None), \'pytest_calls\': len(test_events),\n                   \'junit\': parsed, \'official_junit_ok\': official_ok, \'official_junit_reason\': official_reason,\n                   \'required_tests\': required, \'required_failure_matched\': matched,\n                   \'baseline_commit_exit_code\': git_setup.get(\'baseline_commit_exit_code\'),\n                   \'snapshot_head_matches\': git_setup.get(\'snapshot_head_matches\', False),\n                   \'snapshot_identity_verified\': bool(workspace_setup and workspace_setup[\'status\'] == \'WORKSPACE_READY\'\n                       and git_setup.get(\'native_snapshot_head_unchanged\')\n                       and snapshot_hash == config[\'snapshot_pins\'].get(task_id)),\n                   \'workspace_setup\': workspace_setup,\n                   \'import_origin\': import_origin,\n                   \'duration_seconds\': round(time.perf_counter() - started, 3), \'model_calls\': 0,\n                   \'environment_profile\': \'official_subprocess_pinned_published_snapshot_and_source_path\',\n                   \'raw_reference_patch_exported\': False, \'raw_test_patch_exported\': False}\n        # Never serialize TaskResult wholesale: its agent_patch contains the reference fix.\n        result_output = result.test_output or \'\'\n        if reference_patch.strip() and reference_patch.strip() in result_output:\n            result_output = result_output.replace(reference_patch.strip(), \'[REFERENCE PATCH REMOVED]\')\n        if task.test_patch.strip() and task.test_patch.strip() in result_output:\n            result_output = result_output.replace(task.test_patch.strip(), \'[VERIFICATION PATCH REMOVED]\')\n        (artifact_dir / \'verifier_output.log\').write_text(result_output, encoding=\'utf-8\')\n        if task_id == \'requests_6757\':\n            root = Path(workspace_setup[\'source_root\']) if workspace_setup else None\n            compat_path = root / \'requests/compat.py\' if root else None\n            compat_text = compat_path.read_text() if compat_path and compat_path.is_file() else \'\'\n            contract = {\n                \'phase\': phase,\n                \'snapshot_compat_mentions_is_urllib3_1\': \'is_urllib3_1\' in compat_text,\n                \'reference_patch_mentions_is_urllib3_1\': \'is_urllib3_1\' in reference_patch,\n                \'reference_changed_paths\': re.findall(r\'^\\+\\+\\+ b/(.+)$\', reference_patch, re.MULTILINE),\n                \'test_patch_mentions_is_urllib3_1\': \'is_urllib3_1\' in task.test_patch,\n                \'note\': \'Diagnostic contract check only; no library or test code was changed to supply a missing symbol.\'}\n            write_json(artifact_dir / \'requests_contract.json\', contract)\n        payload[\'compatibility_profile\'] = config.get(\'dependency_overlays\', {}).get(task_id)\n        write_json(artifact_dir / \'result.json\', payload)\n        print(json.dumps({\'instance_id\': task_id, \'phase\': phase, \'resolved\': payload[\'resolved\'],\n                          \'exit_code\': payload[\'test_exit_code\'], \'junit\': {k:v for k,v in parsed.items() if k != \'failed_nodes\'}}), flush=True)\n        return payload\n    finally:\n        manager.close()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', required=True)\n    parser.add_argument(\'--preflight\', action=\'store_true\')\n    parser.add_argument(\'--task-id\')\n    parser.add_argument(\'--phase\', choices=[\'baseline\', \'reference\'])\n    args = parser.parse_args()\n    config = json.loads(Path(args.config).read_text())\n    if args.preflight:\n        result = preflight(config)\n        write_json(Path(config[\'artifact_dir\']) / \'runtime_preflight.json\', result)\n        print(\'Runtime import and verifier-source checks passed.\', flush=True)\n    else:\n        try:\n            asyncio.run(run_phase(config, args.task_id, args.phase))\n        except Exception as exc:\n            payload = {\'instance_id\': args.task_id, \'phase\': args.phase,\n                       \'worker_error\': type(exc).__name__ + \': \' + str(exc), \'model_calls\': 0}\n            write_json(Path(config[\'artifact_dir\']) / \'evaluator_only\' / args.task_id / args.phase / \'result.json\', payload)\n            print(json.dumps(payload), flush=True)\n            raise\n\n\nif __name__ == \'__main__\':\n    main()\n'
SNAPSHOT_PINS = {'fastapi_14479': '24f21d72321d3dc6e7a6258e4bee0ffd169d0b449f518a0c72fd141b6989c0b8', 'fastapi_5624': 'afdbc4ad0cd83a28385f0f507eef2518607e0378d2ded690334e70ca10ed25b4', 'requests_6757': 'dd2c178bb35c5aa3ad82ed8b4393fe087da0fa2fe30465bee4a4d3357475a44c', 'rich_2943': '9b7afcf417a93866ce39ec0feec48e8b2448423f098ec28172171f31dd1e05ad', 'rich_3130': '208ec211fdb8819f192660f95d8476ffb43e4de5c068facc7430664306738536', 'httpx_3672': 'd1afdad9b9d433ebc345968ed90c4c401ed82f5fdfd27ab82eeaf92a49d9e2ed'}
WORKER_PATH = RUN_DIR / 'qualification_worker.py'
WORKER_PATH.write_text(WORKER_SOURCE, encoding='utf-8')
DEV_METADATA = {r['instance_id']: {k:r[k] for k in ['repo', 'base_commit']} for r in DEVELOPMENT}
CONFIG = {
    'version': 'GP01C_v1.0', 'run_id': RUN_ID, 'competition_root': str(COMPETITION_ROOT),
    'artifact_dir': str(ARTIFACTS), 'sandbox_dir': str(RUN_DIR / 'sandboxes'),
    'tasks_sha256': 'e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6',
    'verification_source_sha256': '99e2a5fdab2be2b5e0388de29080da9d6e7003cf8b53299cda790464f4ecd2aa',
    'verify_task_parameters': ['docker', 'config', 'task', 'snapshot_path', 'base_snapshot_path',
                               'patch_path', 'fast_path', 'agent_patch', 'agent_error', 'trace', 'start_time'],
    'development_task_ids': list(DEV_METADATA), 'development_metadata': DEV_METADATA,
    'requested_task_ids': TASK_IDS,
    'snapshot_pins': SNAPSHOT_PINS,
    'dependency_overlays': {},
    'snapshot_identity_policy': 'task_named_published_archive_sha256_and_recorded_native_git_identity',
    'auto_download_results': AUTO_DOWNLOAD_RESULTS,
    'command_timeout_seconds': COMMAND_TIMEOUT_SECONDS,
    'verification_timeout_seconds': VERIFICATION_TIMEOUT_SECONDS,
    'phase_wall_timeout_seconds': PHASE_WALL_TIMEOUT_SECONDS,
    'stop_on_infrastructure_hold': STOP_ON_INFRA_HOLD,
    'worker_sha256': file_sha(WORKER_PATH), 'core_pins': CORE_PINS,
    'split_frozen': False, 'model_calls': 0,
}
CONFIG_PATH = RUN_DIR / 'qualification_config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + '\n', encoding='utf-8')
write_json('03_run_manifest.json', CONFIG)
if not set(TASK_IDS).issubset(DEV_METADATA) or len(set(TASK_IDS)) != len(TASK_IDS):
    HOLDS.append('TASK_IDS must contain unique IDs drawn only from the six development candidates.')
if not HOLDS:
    probe = run_logged([str(VENV_PYTHON), '-u', str(WORKER_PATH), '--config', str(CONFIG_PATH), '--preflight'],
                       'runtime_preflight.log', 300)
    if probe['returncode'] == 0 and (ARTIFACTS / 'runtime_preflight.json').is_file():
        RUNTIME_READY = True
    else:
        HOLDS.append('Evaluator import/source-contract preflight failed. See runtime_preflight.log.')
observe(3, 'Checked the official verifier imports, signature, source hash, and runtime compatibility.',
        runtime_ready=RUNTIME_READY, holds=HOLDS)

if RUNTIME_READY:
    try:
        wheel_dir = RUN_DIR / 'compatibility_wheels'
        wheel_dir.mkdir()
        overlay_dir = RUN_DIR / 'compatibility_profiles' / 'pygments_2_15_1'
        command = [str(VENV_PYTHON), '-m', 'pip', '--disable-pip-version-check', 'download',
                   '--no-index', '--find-links', str(WHEELHOUSE_ROOT),
                   '--find-links', str(COMPETITION_ROOT / 'wheels'),
                   '--no-deps', '--only-binary=:all:', '--dest', str(wheel_dir), 'Pygments==2.15.1']
        downloaded = run_logged(command, 'pygments_download_offline.log', 300)
        if downloaded['returncode'] != 0 and ALLOW_PYPI_BOOTSTRAP:
            command = [arg for arg in command if arg != '--no-index']
            downloaded = run_logged(command, 'pygments_download_pypi.log', 300)
        matches = list(wheel_dir.glob('Pygments-2.15.1-*.whl')) + list(wheel_dir.glob('pygments-2.15.1-*.whl'))
        matches = sorted(set(matches))
        if downloaded['returncode'] != 0 or len(matches) != 1:
            raise RuntimeError('Could not obtain the one pinned Pygments 2.15.1 wheel; inspect the pygments_download logs.')
        wheel = matches[0]
        installed = run_logged([str(VENV_PYTHON), '-m', 'pip', '--disable-pip-version-check', 'install',
                                '--no-index', '--no-deps', '--target', str(overlay_dir), str(wheel)],
                               'pygments_overlay_install.log', 300)
        if installed['returncode'] != 0:
            raise RuntimeError('Pygments overlay installation failed; inspect its log.')
        file_hashes = {relative: file_sha(overlay_dir / relative) for relative in
                       ['pygments/__init__.py', 'pygments/styles/monokai.py', 'pygments/styles/emacs.py', 'pygments/lexers/python.py']}
        profile = {'distribution': 'Pygments', 'version': '2.15.1', 'path': str(overlay_dir),
                   'wheel_name': wheel.name, 'wheel_sha256': file_sha(wheel), 'file_hashes': file_hashes}
        CONFIG['dependency_overlays'] = {'rich_3130': profile}
        CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + '\n', encoding='utf-8')
        write_json('03_run_manifest.json', CONFIG)
        write_json('compatibility_profile.json', profile)
        print('Rich compatibility profile prepared: Pygments 2.15.1, task sandbox only.', flush=True)
    except Exception as exc:
        HOLDS.append(type(exc).__name__ + ': ' + str(exc))
        RUNTIME_READY = False
        print('Compatibility setup held; continue to export diagnostics.', flush=True)


**Observation:** `runtime_ready: true` is required before task execution. Source matching alone is not an execution-qualified task.


## Cell 4 — Run baseline and reference verification in fresh sandboxes


In [ ]:
worker_helpers = runpy.run_path(str(WORKER_PATH), run_name='gp01b_helpers')
classify_pair = worker_helpers['classify_pair']

def run_one_phase(task_id, phase):
    print('START ' + task_id + ' / ' + phase, flush=True)
    outcome = run_logged([str(VENV_PYTHON), '-u', str(WORKER_PATH), '--config', str(CONFIG_PATH),
                          '--task-id', task_id, '--phase', phase],
                         'evaluator_only/' + task_id + '/' + phase + '/worker.log',
                         PHASE_WALL_TIMEOUT_SECONDS)
    result_path = ARTIFACTS / 'evaluator_only' / task_id / phase / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.is_file() else {
        'instance_id': task_id, 'phase': phase, 'worker_error': 'No phase result was produced.'}
    if outcome['returncode'] != 0:
        result['worker_error'] = result.get('worker_error') or ('Worker exit ' + str(outcome['returncode']))
    result['worker_process'] = outcome
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
    print('END ' + task_id + ' / ' + phase + ': exit=' + str(result.get('test_exit_code'))
          + ', resolved=' + str(result.get('resolved')), flush=True)
    return result

if RUNTIME_READY:
    for task_id in TASK_IDS:
        baseline = run_one_phase(task_id, 'baseline')
        baseline_infra = (baseline.get('worker_error') or baseline.get('pytest_calls') != 1
                          or not baseline.get('junit', {}).get('valid_xml')
                          or baseline.get('junit', {}).get('errors', 0) > 0
                          or baseline.get('baseline_commit_exit_code') != 0
                          or not baseline.get('snapshot_identity_verified')
                          or not baseline.get('import_origin', {}).get('inside_snapshot'))
        # The reference phase is also run after a baseline collection error
        # to diagnose the task contract; classify_pair still enforces the same gates.
        reference = run_one_phase(task_id, 'reference')
        decision = classify_pair(baseline, reference)
        row = {'instance_id': task_id, 'repo': DEV_METADATA[task_id]['repo'], **decision,
               'baseline_exit_code': baseline.get('test_exit_code'),
               'reference_exit_code': reference.get('test_exit_code') if reference else None}
        TASK_RESULTS.append(row)
        write_json('04_development_qualification.json', TASK_RESULTS)
        print(json.dumps(row), flush=True)
        if row['status'] == 'INFRA_HOLD' and STOP_ON_INFRA_HOLD:
            HOLDS.append('Stopped additional tasks after infrastructure hold on ' + task_id + '.')
            break
else:
    print('Task execution skipped because the runtime preflight is on hold.', flush=True)

observe(4, 'Recorded baseline/reference outcomes without a model; incomplete tasks remain pending.',
        attempted_tasks=len(TASK_RESULTS), qualified_tasks=sum(r['qualified'] for r in TASK_RESULTS), holds=HOLDS)


**Observation:** Expected qualifying pattern: baseline pytest exit 1 with a matched required-test failure, followed by reference exit 0 with positive verified JUnit. A collection or setup error is not evidence that the bug was reproduced.


## Cell 5 — Record decisions, pending tasks and the next gate


In [ ]:
completed = {r['instance_id'] for r in TASK_RESULTS}
pending = [task_id for task_id in TASK_IDS if task_id not in completed]
pending_reference = [row['instance_id'] for row in TASK_RESULTS if row['reference_exit_code'] is None]
qualified = [r['instance_id'] for r in TASK_RESULTS if r['qualified']]
pytest_executions = sum(json.loads(path.read_text()).get('pytest_calls', 0)
                        for path in (ARTIFACTS / 'evaluator_only').glob('*/*/result.json'))
status = 'DEVELOPMENT_QUALIFICATION_COMPLETE' if len(qualified) == len(TASK_IDS) and not HOLDS else 'REVIEW_REQUIRED'
summary = {
    'version': 'GP01C_v1.0', 'run_id': RUN_ID, 'status': status,
    'runtime_ready': RUNTIME_READY, 'tasks_requested': TASK_IDS,
    'tasks_attempted': len(TASK_RESULTS), 'qualified_development_tasks': qualified,
    'previously_qualified_not_rerun': ['httpx_3672', 'rich_2943', 'fastapi_5624', 'fastapi_14479'],
    'pending_tasks': pending, 'task_outcomes': TASK_RESULTS, 'holds': HOLDS,
    'pending_reference_tasks': pending_reference, 'pytest_executions': pytest_executions,
    'split_frozen': False, 'evaluation_candidates_tested': 0, 'model_calls': 0,
    'reference_repairs_are_agent_results': False,
    'next_step': 'Review execution evidence; resolve infrastructure issues, qualify remaining candidates, then prepare one autonomous repair smoke run.',
}
write_json('00_summary.json', summary)
observe(5, 'Retained every attempted outcome and listed unattempted tasks explicitly.',
        status=status, qualified=qualified, pending=pending)
write_json('05_cell_observations.json', OBSERVATIONS)
notes = ['# GP01C targeted qualification follow-up', '', 'Status: ' + status, '',
         'Model calls: 0. Reference fixes are evaluator controls, not autonomous repairs.', '',
         '| Task | Status | Baseline exit | Reference exit |', '| --- | --- | ---: | ---: |']
for row in TASK_RESULTS:
    notes.append('| ' + row['instance_id'] + ' | ' + row['status'] + ' | ' + str(row['baseline_exit_code']) + ' | ' + str(row['reference_exit_code']) + ' |')
notes += ['', 'Unattempted tasks: ' + (', '.join(pending) or 'none'),
          'Pending reference phases: ' + (', '.join(pending_reference) or 'none'),
          'Pytest executions: ' + str(pytest_executions), '', '## Holds', '']
notes += ['- ' + h for h in HOLDS] if HOLDS else ['No global infrastructure holds; inspect individual task outcomes.']
notes += ['', '## Boundaries', '',
          'The official subprocess verifier used pinned task-named published archives and snapshot source-origin checks. Original commit ID equality and Docker equivalence are not claimed.',
          'Files under evaluator_only contain protected test diagnostics. Keep them out of agent inputs.',
          'No final evaluation split, model configuration or research outcome is frozen.', '',
          '## Next', '', summary['next_step'], '']
(ARTIFACTS / '06_observations.md').write_text('\n'.join(notes), encoding='utf-8')
print('RESULT:', status, '| qualified:', len(qualified), '| pending:', len(pending))


**Observation:** This follow-up records the two remaining cases; four prior qualified tasks are retained without rerun. Partial results remain useful; no task is silently replaced or counted as a repair success.


## Cell 6 — Export the review artifacts and request a browser download


In [ ]:
"""Best-effort interactive notebook download with explicit manual fallbacks."""

def offer_zip_download(bundle, auto_download=True, embed_limit_mb=10):
    from pathlib import Path
    import base64
    import html
    import json
    import os
    from IPython.display import FileLink, HTML, Javascript, display

    bundle = Path(bundle).resolve()
    relative = os.path.relpath(bundle, Path.cwd())
    print('ZIP ready:', bundle)
    print('If the browser does not start a download, use the Download ZIP button or Kaggle Output panel.')
    display(FileLink(relative))
    if bundle.stat().st_size > embed_limit_mb * 1024 * 1024:
        print('This ZIP exceeds the inline-download size limit; use the file link or Output panel.')
        return {'automatic_download_requested': False, 'download_confirmed': False, 'reason': 'size_limit'}

    encoded = base64.b64encode(bundle.read_bytes()).decode('ascii')
    name = html.escape(bundle.name, quote=True)
    button = ('<a download="' + name + '" href="data:application/zip;base64,' + encoded + '" '
              'style="display:inline-block;padding:12px 20px;background:#174EA6;color:white;'
              'text-decoration:none;border-radius:6px;font-weight:600">Download ZIP</a>')
    display(HTML(button))
    if auto_download:
        script = '''(function () {
  try {
    const payload = __PAYLOAD__;
    const bytes = Uint8Array.from(atob(payload), function (c) { return c.charCodeAt(0); });
    const url = URL.createObjectURL(new Blob([bytes], {type: "application/zip"}));
    const link = document.createElement("a");
    link.href = url;
    link.download = __FILENAME__;
    link.style.display = "none";
    document.body.appendChild(link);
    link.click();
    link.remove();
    setTimeout(function () { URL.revokeObjectURL(url); }, 60000);
  } catch (error) {
    console.info("Automatic ZIP download was not available; use the Download ZIP button or Output panel.", error);
  }
})();'''.replace('__PAYLOAD__', json.dumps(encoded)).replace('__FILENAME__', json.dumps(bundle.name))
        display(Javascript(script))
        print('Automatic browser-download request offered. Notebook code cannot confirm that your PC saved it.')
    return {'automatic_download_requested': bool(auto_download), 'download_confirmed': False,
            'fallbacks': ['Download ZIP button', 'file link', 'Kaggle Output panel']}

# The runtime virtualenv, extracted snapshots and temporary patch files are outside
# ARTIFACTS and deliberately excluded from the result package.
bundle = RUN_DIR / 'GP01C_results.zip'
files = sorted(p for p in ARTIFACTS.rglob('*') if p.is_file())
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(ARTIFACTS).as_posix())
print('DOWNLOAD:', bundle)
print('Return this ZIP, including when the result is REVIEW_REQUIRED.')
try:
    download_request = offer_zip_download(bundle, auto_download=AUTO_DOWNLOAD_RESULTS)
except Exception as exc:
    print('Browser-download display was unavailable:', type(exc).__name__)
    print('The ZIP is still saved at the path above. Download it from the Kaggle Output panel.')


**Observation:** Return `GP01C_results.zip`. If the browser does not start the download, use the Download ZIP button, file link or Output panel. The archive contains evaluator-only test evidence and environment diagnostics, without model weights, snapshots, virtual environments or intentionally exported raw patches.
